<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        🌳 03. Métodos de Ensamble: Bagging y Random Forests
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 09
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/09%20-%20Decision%20Trees/Para%20Dummies/03_Metodos_Ensamble_Bagging_y_Random_Forests_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

En el cuaderno anterior vimos que un solo árbol puede **memorizar** los datos (sobreajuste) y que podarlo ayuda, pero no siempre es suficiente. ¿Y si en vez de confiar en un solo árbol, le preguntamos a **muchos árboles distintos** y combinamos sus respuestas?

Esa es la idea central de este cuaderno. Verás, con ejemplos pequeños:

1. Por qué "preguntarle a un comité" suele ser mejor que preguntarle a una sola persona.
2. Qué es **Bagging** (entrenar muchos árboles con datos "revueltos" al azar y promediar sus votos).
3. Qué es un **Random Forest** (Bosque Aleatorio) y en qué se diferencia de Bagging.
4. Cómo saber qué variables usó más el bosque para decidir (`feature_importances_`).

---
## 1. La sabiduría del comité 👥

Imagina que quieres saber si una fruta está madura. Le puedes preguntar a **una sola persona** — quizás acierte, quizás no, depende de su ojo entrenado ese día. O le puedes preguntar a **20 personas distintas**, cada una mirando la fruta desde un ángulo o con una experiencia distinta, y quedarte con lo que diga la mayoría.

Ese segundo enfoque casi siempre es más confiable: los errores de unas personas se compensan con los aciertos de otras, siempre y cuando cada persona opine "por su cuenta" y no todas copien la misma respuesta.

Esa es la idea de un **ensamble (*ensemble*)** en Machine Learning: en vez de un solo árbol de decisión, entrenamos **muchos árboles distintos** y combinamos sus predicciones — por votación (clasificación) o por promedio (regresión).

> 📌 **Para recordar:** un árbol solo puede equivocarse por casualidad. Un comité de árboles diversos que vota en conjunto se equivoca mucho menos, porque sus errores individuales tienden a cancelarse.

---
## 2. Bagging: muchos árboles, cada uno con su propia "muestra revuelta" 🎲

¿Cómo logramos que los árboles del comité sean **distintos** entre sí, si todos ven básicamente los mismos datos? La respuesta es el truco de **Bagging** (*Bootstrap Aggregating*):

1. A cada árbol del comité le damos, no el dataset completo, sino una **muestra al azar y con reemplazo** del mismo tamaño (esto se llama *bootstrap*). "Con reemplazo" significa que un mismo dato puede aparecer repetido en la muestra de un árbol, y no aparecer para nada en la de otro.
2. Cada árbol crece sin restricciones sobre su propia muestra revuelta.
3. Al final, le preguntamos a **todos** los árboles y combinamos sus respuestas (voto mayoritario).

Es como repartir la misma encuesta a 100 encuestadores, pero cada uno solo entrevista a un subgrupo de personas elegido al azar (con la posibilidad de repetir a alguien). Ningún encuestador ve exactamente lo mismo que otro, así que sus conclusiones individuales varían un poco — y por eso, al promediarlas, el resultado final es más estable que el de un solo encuestador.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
import warnings
warnings.filterwarnings('ignore')

np.random.seed(7)

print("Entorno listo para explorar Bagging y Random Forests.")

---
## 3. Creamos un mini-set de "frutas maduras o no" 🍎

Inventamos 200 frutas de muestra con dos características fáciles de imaginar: `dulzura` (qué tan dulce es, en una escala) y `firmeza` (qué tan firme está al tacto). La etiqueta `madura` indica si la fruta está lista para comer (1) o no (0). Usamos una regla con algo de ruido para que el problema no sea trivial — como pasa en la vida real, no siempre hay una frontera perfecta.

In [ ]:
n = 200
dulzura = np.random.uniform(0, 10, n)
firmeza = np.random.uniform(0, 10, n)

# Regla base: una fruta tiende a estar madura si es dulce y no demasiado firme,
# con algo de ruido para que no sea un problema perfecto
puntaje = dulzura - 0.6 * firmeza + np.random.normal(0, 1.5, n)
madura = (puntaje > 2).astype(int)

frutas = pd.DataFrame({'dulzura': dulzura, 'firmeza': firmeza, 'madura': madura})

print(frutas['madura'].value_counts())
frutas.head(8)

### 🤔 ¿Qué acaba de pasar?

- Generamos 200 frutas al azar, cada una con un valor de `dulzura` y `firmeza`.
- `puntaje` combina ambas variables con una regla simple más ruido aleatorio (`np.random.normal`), simulando que en la vida real ninguna regla es perfecta.
- `madura = (puntaje > 2).astype(int)` convierte ese puntaje en una etiqueta de 0 (no madura) o 1 (madura), que es justo lo que un árbol de clasificación intentará predecir.
- `value_counts()` nos deja ver cuántas frutas quedaron en cada clase — es un buen hábito revisarlo siempre antes de entrenar cualquier modelo.

---
## 4. Comparación: un solo árbol vs. Bagging vs. Random Forest 🌳🌳🌳

Separamos las frutas en un grupo de entrenamiento y uno de prueba, y entrenamos tres modelos:

1. **Un solo árbol de decisión** (nuestro punto de partida, el "encuestador único").
2. **Bagging**: 100 árboles, cada uno entrenado con su propia muestra bootstrap.
3. **Random Forest**: también 100 árboles, pero con un truco extra que veremos después de correr el código.

Comparemos qué tan bien predicen frutas que **nunca vieron** (el conjunto de prueba).

In [ ]:
X = frutas[['dulzura', 'firmeza']]
y = frutas['madura']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# 1. Un solo árbol
arbol_unico = DecisionTreeClassifier(random_state=42)
arbol_unico.fit(X_train, y_train)

# 2. Bagging: 100 árboles, cada uno con su propia muestra "revuelta"
bagging = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100, random_state=42)
bagging.fit(X_train, y_train)

# 3. Random Forest: 100 árboles con una aleatoriedad extra
bosque = RandomForestClassifier(n_estimators=100, random_state=42)
bosque.fit(X_train, y_train)

print(f"Exactitud - Un solo árbol:  {arbol_unico.score(X_test, y_test)*100:.1f}%")
print(f"Exactitud - Bagging:        {bagging.score(X_test, y_test)*100:.1f}%")
print(f"Exactitud - Random Forest:  {bosque.score(X_test, y_test)*100:.1f}%")

### 🤔 ¿Qué acaba de pasar?

- Entrenamos los tres modelos sobre las mismas frutas de entrenamiento, y los evaluamos con las mismas frutas de prueba (que ninguno vio durante el entrenamiento).
- `BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=100)` crea 100 árboles, cada uno entrenado con una muestra bootstrap distinta, y combina sus votos.
- `RandomForestClassifier(n_estimators=100)` también crea 100 árboles, pero por dentro es un "Bagging con un ingrediente extra" que explicamos en la siguiente sección.
- Con datasets pequeños como este, las tres exactitudes pueden quedar parecidas — la ventaja real de Bagging y Random Forest se nota sobre todo con datos más ruidosos o más grandes, donde un solo árbol tiende a sobreajustarse mucho más. Lo importante aquí es entender **el mecanismo**, no solo el número final.

---
## 5. El ingrediente extra de Random Forest: no dejar que un árbol "domine" el comité 🔁

Imagina que en tu comité de 100 personas, todas tienden a fijarse primero en la **misma característica** (por ejemplo, todas miran primero la dulzura antes que cualquier otra cosa). Si eso pasa, tus 100 "opiniones" en realidad no son tan distintas entre sí — todas parten del mismo primer criterio, así que sus errores se parecen y no se cancelan tan bien.

**Random Forest** evita esto con una regla adicional: en cada decisión que toma un árbol dentro del bosque, **no le permite mirar todas las variables disponibles** — solo un subconjunto elegido al azar. Así, unos árboles se ven obligados a fijarse primero en la firmeza, otros en la dulzura, y el comité termina siendo mucho más diverso.

> 📌 **Para recordar:** Bagging revuelve **los datos** que ve cada árbol. Random Forest, además, revuelve **las variables** que cada árbol puede considerar en cada decisión. Esa doble revoltura hace que los árboles del bosque se parezcan menos entre sí, y por eso el comité final suele ser más confiable.

---
## 6. ¿Qué variable usó más el bosque? `feature_importances_` 🔍

Una ventaja práctica de Random Forest es que, además de predecir, nos puede decir **qué tanto usó cada variable** para tomar sus decisiones en todo el bosque. A esto se le llama **importancia de características** (*feature importances*).

Es como preguntarle al comité completo: *"de las pistas que tenían disponibles, ¿cuál revisaron más seguido para decidir si la fruta estaba madura?"*

In [ ]:
importancias = pd.Series(bosque.feature_importances_, index=X.columns)
importancias = importancias.sort_values(ascending=False)

print("Importancia de cada variable según el Random Forest:")
print(importancias)

plt.figure(figsize=(6, 3.5), dpi=110)
importancias.plot(kind='barh', color='#f59e0b')
plt.xlabel('Importancia relativa')
plt.title('¿Qué variable usó más el bosque?', fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

### 🤔 ¿Qué acaba de pasar?

- `bosque.feature_importances_` devuelve un número por cada variable de entrada: entre más alto, más "peso" tuvo esa variable en las decisiones de los 100 árboles del bosque.
- Como construimos `puntaje` dándole más peso a la `dulzura` que a la `firmeza` (`dulzura - 0.6 * firmeza`), es razonable esperar que el bosque también encuentre a la dulzura como la variable más importante — aunque con datos ruidosos esto no siempre sale exacto.
- Esta herramienta es muy usada en la práctica: con datasets de decenas o cientos de columnas, `feature_importances_` ayuda a identificar rápidamente cuáles variables realmente le importan al modelo, sin tener que mirar los 100 árboles uno por uno.

---
## 7. Resumen relámpago ⚡

| Idea | En una frase |
|---|---|
| Ensamble (*ensemble*) | Combinar las predicciones de varios modelos en vez de confiar en uno solo — la "sabiduría del comité". |
| Bagging (*Bootstrap Aggregating*) | Entrena muchos árboles, cada uno con su propia muestra de datos elegida al azar y con reemplazo. |
| Random Forest | Bagging + un truco extra: en cada decisión, cada árbol solo puede mirar un subconjunto al azar de las variables, para que los árboles se parezcan menos entre sí. |
| `BaggingClassifier` / `RandomForestClassifier` | Las clases de Scikit-Learn que implementan estas ideas, entrenando muchos `DecisionTreeClassifier` por debajo. |
| `feature_importances_` | Le dice qué tanto usó el bosque cada variable para tomar sus decisiones — útil para entender el modelo. |

➡️ **Siguiente paso:** en el cuaderno [04 - Boosting y Casos de Estudio Desbalanceados (Para Dummies)](04_Boosting_y_Casos_Estudio_Desbalanceados_Dummies.ipynb) descubrirás otra forma de combinar árboles — esta vez, entrenándolos **uno después de otro**, cada uno aprendiendo de los errores del anterior — y verás cómo manejar problemas donde una clase es mucho más rara que la otra.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos (Edición Para No Ingenieros)</i>
  </p>
</div>